#Please note - I have changed and tested with various parameters. Also need to update to all the directories and paths for data where needed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

#add path
ZIP_PATH = ""
EXTRACT_PATH = ""

if not os.path.exists(EXTRACT_PATH):
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    !unzip -q {ZIP_PATH} -d {EXTRACT_PATH}

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    TimeDistributed,
    GlobalAveragePooling2D,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
CSV_FILE = "/content/CE_Labels.csv"
df = pd.read_csv(CSV_FILE)

IMG_SIZE = 224
NUM_FRAMES = 16
BATCH_SIZE = 8 # we can try differnt parameters
EPOCHS = 50
FRAMES_DIR = ""   


In [ ]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["engagement_label"])

num_classes = len(label_encoder.classes_)

attention net

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['engagement_label'])

In [ ]:


class VideoSequenceGenerator(Sequence):
    def __init__(self, dataframe, batch_size):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))


    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch_df.iterrows():
            video_folder = os.path.join(FRAMES_DIR, row["video_filename"])
            all_frames = sorted(os.listdir(video_folder))

            total_available = len(all_frames)
            if total_available >= NUM_FRAMES:
                indices = np.linspace(0, total_available - 1, NUM_FRAMES).astype(int)
                selected_frames = [all_frames[i] for i in indices]
            else:
                selected_frames = all_frames

            video_frames = []
            for frame_name in selected_frames:
                frame_path = os.path.join(video_folder, frame_name)
                img = cv2.imread(frame_path)
                if img is None: continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0
                video_frames.append(img)

            while len(video_frames) < NUM_FRAMES:
                video_frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

            X.append(video_frames)
            y.append(row["label"])

        return np.array(X), tf.keras.utils.to_categorical(y, num_classes=3)

In [ ]:
train_gen = VideoSequenceGenerator(train_df, batch_size=4)
val_gen = VideoSequenceGenerator(val_df, batch_size=4)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_stable_attention_net():
    video_input = layers.Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3))

    base_model = tf.keras.applications.MobileNetV2(
        weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    x = layers.TimeDistributed(base_model)(video_input)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    score = layers.Dense(1, activation='tanh')(x)
    attention_weights = layers.Activation('softmax')(score)

    context = layers.Multiply()([x, attention_weights])

    x = layers.Bidirectional(layers.LSTM(64))(context)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(3, activation="softmax")(x)

    return Model(video_input, output)

tf.keras.backend.clear_session()
model = build_stable_attention_net()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

In [ ]:
from sklearn.utils import class_weight
import numpy as np


weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['engagement_label']),
    y=train_df['engagement_label']
)

class_weights_dict = dict(enumerate(weights))

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    class_weight=class_weights_dict,
    callbacks=callbacks
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


y_pred_probs = model.predict(val_gen)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = []
for i in range(len(val_gen)):
    _, labels = val_gen[i]
    y_true.extend(np.argmax(labels, axis=1))

y_true = np.array(y_true)

target_names = ['Low', 'Medium', 'High']
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('AttentionNet: Classroom Engagement Confusion Matrix')
plt.show()

In [ ]:
model.save('AttentionNet_Classroom_Final.keras')

model.save_weights('AttentionNet_Weights_Only.h5')


In [ ]:
from google.colab import files

files.download('AttentionNet_Classroom_Final.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>